In [2]:
# ==========================================================
# WEEK 6 BBO OPTIMISATION USING A CNN-INSPIRED 1D MODEL
# ==========================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# ==========================================================
# WEEK 1–5 INPUT DATA
# ==========================================================

inputs_by_week = [
    [
        np.array([0.211325, 0.788675]),
        np.array([0.723607, 0.276393]),
        np.array([0.166667, 0.500000, 0.833333]),
        np.array([0.125000, 0.375000, 0.625000, 0.875000]),
        np.array([0.875000, 0.625000, 0.375000, 0.125000]),
        np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
        np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
        np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
    ],
    [
        np.array([0.654299, 0.353479]),
        np.array([0.754299, 0.253479]),
        np.array([0.304299, 0.553479, 0.709691]),
        np.array([0.804299, 0.603479, 0.409691, 0.205016]),
        np.array([0.854299, 0.653479, 0.359691, 0.155016]),
        np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
        np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
        np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.712865, 0.284413]),
        np.array([0.118496, 0.481282, 0.876608]),
        np.array([1.000000, 0.683447, 0.334333, 0.000000]),
        np.array([0.882245, 0.615032, 0.380358, 0.114494]),
        np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
        np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
        np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
    ],
    [
        np.array([0.183704, 0.127166]),
        np.array([0.255353, 0.749841]),
        np.array([0.094906, 0.770362, 0.859589]),
        np.array([0.820856, 0.226975, 0.784625, 0.113609]),
        np.array([0.917860, 0.273128, 0.475575, 0.052446]),
        np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
        np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
        np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
    ],
    [
        np.array([0.582812, 0.421077]),
        np.array([0.684812, 0.281383]),
        np.array([0.171685, 0.508728, 0.819757]),
        np.array([0.734042, 0.687625, 0.466427, 0.303264]),
        np.array([0.568707, 0.909615, 0.957143, 0.011420]),
        np.array([0.744968, 0.084644, 0.801527, 0.169728, 0.990511]),
        np.array([1.000000, 0.229371, 0.690054, 0.244152, 0.521424, 1.000000]),
        np.array([0.022620, 0.000000, 0.288709, 0.762817, 0.551656, 0.961269, 0.843087, 1.000000])
    ]
]

# ==========================================================
# WEEK 1–5 OUTPUT DATA
# ==========================================================

outputs_by_week = [
    np.array([
        1.1327056288856873e-125, 0.5675786892822564,
        -0.032385408076210126, -17.20744048260943,
        60.223125, -1.3287857969718009,
        0.34356041660314957, 9.0501517903694
    ]),
    np.array([
        1.1867858443665185e-41, 0.2715258567299176,
        -0.1198581070659559, -13.082213230390916,
        50.179390256321376, -1.5080002951000964,
        0.31639485635485903, 9.0219949134074
    ]),
    np.array([
        7.99676981308551e-19, 0.5213723465552891,
        -0.04726977098841539, -25.67625344929702,
        64.78245026474816, -1.7372122723701597,
        0.3507828450585503, 9.0587238074074
    ]),
    np.array([
        -2.410121917336285e-100, 0.0244939290195959,
        -0.04207093453964322, -20.330739763644917,
        61.278397876784794, -2.4624737429462145,
        0.0634803557047261, 8.275283250689899
    ]),
    np.array([
        7.99676981308551e-19, 0.4258127251524913,
        -0.06232295859499999, -12.496845976106417,
        942.2521944777988, -2.280900502246122,
        0.21828066675598462, 8.427686115001
    ])
]

# ==========================================================
# CNN SURROGATE MODEL
# ==========================================================

class CNN1DSurrogate(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=8, kernel_size=2, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=8, out_channels=16, kernel_size=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)   # shape: batch, channel, features
        x = self.conv(x)
        return self.fc(x)

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def get_function_history(fn_idx):
    X = np.vstack([week[fn_idx] for week in inputs_by_week]).astype(np.float32)
    y = np.array([week[fn_idx] for week in outputs_by_week], dtype=np.float32)
    return X, y

def standardize_y(y):
    mean = y.mean()
    std = y.std()
    if std < 1e-8:
        std = 1.0
    return (y - mean) / std, mean, std

def train_cnn_model(X, y, epochs=1500, lr=0.01):
    y_scaled, y_mean, y_std = standardize_y(y)

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32)

    model = CNN1DSurrogate(input_dim=X.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        predictions = model(X_tensor)
        loss = loss_fn(predictions, y_tensor)

        loss.backward()
        optimizer.step()

    return model, y_mean, y_std

def predict(model, candidates, y_mean, y_std):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(candidates, dtype=torch.float32)
        preds = model(X_tensor).numpy().ravel()
    return preds * y_std + y_mean

def generate_candidates(X, y, n_local=6000, n_global=2000):
    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_x = X[best_idx]

    second_idx = np.argsort(y)[-2]
    second_x = X[second_idx]

    # Local search around best historical input
    local_best = np.clip(best_x + np.random.normal(0, 0.05, size=(n_local, dim)), 0, 1)

    # Local search around second-best input
    local_second = np.clip(second_x + np.random.normal(0, 0.08, size=(n_local // 2, dim)), 0, 1)

    # Interpolation between best and second-best
    mix = np.random.uniform(0, 1, size=(1000, 1))
    interp = np.clip(mix * best_x + (1 - mix) * second_x + np.random.normal(0, 0.02, size=(1000, dim)), 0, 1)

    # Global exploration
    global_search = np.random.uniform(0, 1, size=(n_global, dim))

    return np.vstack([local_best, local_second, interp, global_search, X]), best_x

def propose_week6(fn_idx):
    X, y = get_function_history(fn_idx)

    model, y_mean, y_std = train_cnn_model(X, y)

    candidates, best_x = generate_candidates(X, y)

    preds = predict(model, candidates, y_mean, y_std)

    # Penalise candidates too far from the current best
    distance = np.linalg.norm(candidates - best_x, axis=1)
    score = preds - 0.04 * distance

    best_candidate = candidates[np.argmax(score)]

    return np.round(best_candidate, 6), np.max(y), X[np.argmax(y)]

# ==========================================================
# GENERATE WEEK 6 INPUTS
# ==========================================================

week6_inputs = []

print("CNN-Inspired Week 6 Inputs")
print("==========================")

for i in range(8):
    x6, best_output, best_input = propose_week6(i)
    week6_inputs.append(x6)

    print(f"\nFunction {i+1}")
    print(f"Best output so far: {best_output}")
    print(f"Best input so far:  {np.round(best_input, 6)}")
    print(f"Week 6 input:       {x6}")

print("\nPortal-ready Week 6 format")
print("==========================")

for i, arr in enumerate(week6_inputs, start=1):
    portal_format = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {portal_format}")

CNN-Inspired Week 6 Inputs

Function 1
Best output so far: 7.996769605832161e-19
Best input so far:  [0.582812 0.421077]
Week 6 input:       [0.582812 0.421077]

Function 2
Best output so far: 0.5675786733627319
Best input so far:  [0.723607 0.276393]
Week 6 input:       [0.996744 0.999561]

Function 3
Best output so far: -0.032385408878326416
Best input so far:  [0.166667 0.5      0.833333]
Week 6 input:       [0.166667 0.5      0.833333]

Function 4
Best output so far: -12.496846199035645
Best input so far:  [0.734042 0.687625 0.466427 0.303264]
Week 6 input:       [0.743137 0.718436 0.58181  0.365984]

Function 5
Best output so far: 942.252197265625
Best input so far:  [0.568707 0.909615 0.957143 0.01142 ]
Week 6 input:       [0.396883 0.999813 0.995541 0.905102]

Function 6
Best output so far: -1.32878577709198
Best input so far:  [0.15 0.35 0.55 0.75 0.95]
Week 6 input:       [0.236209 0.379873 0.584786 0.843391 1.      ]

Function 7
Best output so far: 0.3507828414440155
Best inp